# 15b · 加上我们的能量，生成器会更准吗

## 这一章在整个项目里的位置

```
第三阶段 G
  ├─ 15  前提查验 ✅
  ├─ 15a 生成器闸门 ✅
  ├─ 15c 该固定哪些位置 ✅（位置级）
  ├─ 15b 逐个变体的消融  ← 你在这里
  ├─ 15d 平台化
  └─ 15e 讲法
```

## 这一章回答一个问题

15c 是在**位置**这个粒度上比的。这一章下到**每一个具体突变**。

问题只有一个，而且是整个第三阶段绕不过去的：

> **生成器自己已经见过这些考试蛋白的结构了（它的预训练数据里有）。
> 那我们的能量到底还有没有加进去的价值？**

这就是第 15 章列的**第 3 号泄漏面**——我们的切分保护的是打分器，**保护不了生成器**。
唯一的解药就是这个消融。

## 怎么让生成器给单个突变打分

ProteinMPNN 本来是用来写整条序列的，但它每个位置会输出 20 种氨基酸的概率。
所以一个突变的得分就是：

```
log P(换成的那个) − log P(原来那个)
```

负得越多 = 这个替换越不像话。**这是「只用生成器」的基线。**

## 组合方式不许拟合

把两个分数各自在实验内部转成**排名**再平均。**没有任何参数**，所以不存在在考试集上调参。

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)
big = pd.read_parquet("data/models/variant_ablation.parquet")
print(f"{big.dataset_id.nunique()} 组实验 | {big.uniprot_id.nunique()} 个蛋白 | {len(big):,} 个变体")

---

## 训练用的是干净版

模型在 **train split** 的蛋白上训练，而且 Tsuboyama 那部分**去掉了 52 个和考试蛋白同源的结构域**
——这份名单由 `scripts/find_related.py` 重算，结果和 14a 公布的一致（56 个同源，52 个在训练侧）。

训练数据的构成：

| 类型 | 行数 |
|---|---|
| 测蛋白量 | 469,829 |
| 测活性 | 360,978 |
| kcal/mol 稳定性 | 193,600 |
| 稳定性分数 | 10,593 |

## 结果

In [ ]:
def ranks(s):
    return stats.rankdata(s) / len(s)


SCORES = {
    "只用生成器 (ProteinMPNN)": lambda g: g.mpnn,
    "只用我们 两个能量": lambda g: g.ours_both,
    "只用我们 折叠能": lambda g: g.ours_fold,
    "只用我们 功能能": lambda g: g.ours_func,
    "ESM-2 (零样本)": lambda g: g.esm_delta,
    "生成器 + 我们两个能量": lambda g: (ranks(g.mpnn) + ranks(g.ours_both)) / 2,
    "生成器 + ESM-2": lambda g: (ranks(g.mpnn) + ranks(g.esm_delta)) / 2,
}

per_assay = {n: np.array([stats.spearmanr(g.label, f(g)).statistic
                          for _, g in big.groupby("dataset_id")]) for n, f in SCORES.items()}
print(pd.DataFrame([{"打分方式": n, "中位 Spearman": round(float(np.median(v)), 3)}
                    for n, v in per_assay.items()])
      .sort_values("中位 Spearman", ascending=False).to_string(index=False))

In [ ]:
base = per_assay["只用生成器 (ProteinMPNN)"]
print("配对检验，对手是「只用生成器」：\n")
for name in ("生成器 + 我们两个能量", "只用我们 两个能量", "生成器 + ESM-2"):
    a = per_assay[name]
    print(f"  {name:<20} 中位 {np.median(a):.3f} vs {np.median(base):.3f} | "
          f"更好 {int((a > base).sum())}/{len(a)} 组 | p = {stats.wilcoxon(a, base).pvalue:.1e}")

### 结果怎么看

| 打分方式 | 中位 Spearman |
|---|---|
| 只用我们 折叠能 | **0.621** |
| 生成器 + 我们两个能量 | 0.612 |
| 只用我们 两个能量 | 0.609 |
| 生成器 + ESM-2 | 0.527 |
| **只用生成器** | **0.496** |
| 只用我们 功能能 | 0.495 |
| ESM-2 零样本 | 0.415 |

**消融的答案（配对检验）：**

| 对比 | 结果 |
|---|---|
| 生成器 **+ 我们的能量** vs 只用生成器 | 0.612 vs 0.496，**130/132 组更好，p = 2.5e-23** |
| 生成器 **+ ESM-2** vs 只用生成器 | 0.527 vs 0.496，91/132 组，p = 2.3e-08 |

**加我们的能量，效果远大于加 ESM-2。** 第 3 号泄漏面有了答案：
生成器见过这些蛋白的结构，**但我们的能量仍然带来了它没有的信息**。

### 一条必须直说的

**我们的折叠能单独用（0.621）比组合（0.612）还略好。**

也就是说，在"**给已有突变排序**"这个任务上，**生成器没有提供我们没有的东西**。

但这不等于生成器没用——**排序和设计是两回事**：

| | 排序 | 设计 |
|---|---|---|
| 要做什么 | 给**已经列出来的**突变打分 | **凭空写出**新序列 |
| 我们的模型 | 能做，而且更好 | **做不了**，它只会打分 |
| 生成器 | 能做，但差一截 | **只有它能做** |

**所以分工是清楚的：生成器出方案，我们的能量来挑。**
这一章证明的是后半句——挑这件事，我们的能量确实管用。

### 还有一个和 15c 一致的限制

132 组里大部分仍是测蛋白量的，所以折叠能占便宜。功能能单独只有 0.495，
和生成器（0.496）打平——**功能那一半的优势要到 15c 的 KRAS 功能轴上才看得出来**。

## 泄漏自查

| 检查项 | 结果 |
|---|---|
| 训练蛋白 | 只有 train split |
| Tsuboyama 去污染 | ✅ 去掉 52 个同源结构域，和 14a 一致 |
| 组合方式有没有拟合 | ✅ 排名平均，零参数 |
| 实测标签 | 只用于最后评分，没参与训练或调参 |
| 生成器的预训练暴露 | **有**，它见过考试蛋白的结构——**这正是本章要量化的东西** |

## 记账

数字由本 notebook 从 `data/models/variant_ablation.parquet` 重算，
该文件由 `scripts/chapter15b.py` 生成（含训练）。